<a href="https://colab.research.google.com/github/goderdzi/goderdzi/blob/main/notebooks/piper_model_exporter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# <font color="ffc800"> **[Piper](https://github.com/rhasspy/piper) model exporter.**
## ![Piper logo](https://contribute.rhasspy.org/img/logo.png)
---

* Notebook created by: [rmcpantoja](http://github.com/rmcpantoja)
* Collaborator: [Xx_Nessu_xX](http://github.com/XxNessuxX)

In [7]:
#@markdown # <font color="ffc800"> **Install software.** 📦
#@markdown ---

print("\033[93mInstalling...")
%cd /content
!git clone -q https://github.com/rmcpantoja/piper
%cd /content/piper/src/python
!pip install -q cython>=0.29.0 numpy>=1.26 librosa>=0.9.2 onnx onnxruntime-gpu pytorch_lightning onnxscript
!bash build_monotonic_align.sh
!pip install -q --upgrade --force-reinstall gdown

# Fix for PyTorch 2.6+ weights_only security error
import torch
import pathlib
torch.serialization.add_safe_globals([pathlib.PosixPath])

print("\033[93mDone!")

Installing...
/content
fatal: destination path 'piper' already exists and is not an empty directory.
/content/piper/src/python
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.
Done!


In [4]:
import sys
!{sys.executable} -m pip install -U gdown

In [8]:
#@markdown # <font color="ffc800"> **Voice package generation section.** 🗣️
#@markdown ---
%cd /content/piper/src/python
import os
import json
import re
import pathlib
import torch
import ipywidgets as widgets
from IPython.display import display
from google.colab import output

# Registration for the current session
torch.serialization.add_safe_globals([pathlib.PosixPath])

guideurl = "https://github.com/rmcpantoja/piper/blob/master/notebooks/wav/en"
#@markdown #### *Download:*
#@markdown **Drive ID or direct download link of the model in another cloud:**
model_id = "https://drive.google.com/file/d/1NtbcuPY6tw3BFWhJcZPpqSmOGR6NH3KQ/view?usp=sharing" #@param {type:"string"}
#@markdown **Drive ID or direct download link of the config.json file:**
config_id = "https://drive.google.com/file/d/1rC-9ALvlN9wAD5enfRZZf4gwr6ym1l4D/view?usp=sharing" #@param {type:"string"}
#@markdown ---

def get_drive_id(url):
    if "drive.google.com" in url:
        match = re.search(r"[-\w]{25,}", url)
        return match.group(0) if match else url
    return url

#@markdown #### *Creation process:*
#@markdown **Choose the language code (iso639-1 format):**
language = "ka_GE" #@param ["ar_JO", "ca_ES", "cs_CZ", "da_DK", "de_DE", "el_GR", "en_GB", "en_US", "es_ES", "es_LA", "fi_FI", "fr_FR", "grc", "hu_GU", "is_IS", "it_IT", "kk_KZ", "ka_GE", "lb_LU", "nb", "ne", "nl_BE", "no_NO", "pl_PL", "pt_BR", "pt_PT", "ro_RO", "ru_RU", "sk_SK", "sr", "sv_SE", "sw_CD", "tr_TR", "uk_UA", "vi_VN", "zh_CN"]
voice_name = "ucha" #@param {type:"string"}
voice_name = voice_name.lower()
quality = "medium" #@param ["high", "low", "medium", "x-low"]
write_model_card = False #@param {type:"boolean"}
streaming = False #@param {type:"boolean"}

def start_process(streaming):
    if not os.path.exists("/content/project/model.ckpt"):
        raise Exception("Could not download model!")
    output.eval_js(f'new Audio("{guideurl}/starting.wav?raw=true").play()')

    export_mod = "piper_train.export_onnx" if not streaming else "piper_train.export_onnx_streaming"
    output_arg = f"{export_voice_path}/{export_voice_name}.onnx" if not streaming else f"{export_voice_path}"

    print("\033[93mExporting ONNX...")

    # Create a wrapper script to run the export inside the safe_globals context
    wrapper_script = f"""
import pathlib
import torch
import sys
import importlib

torch.serialization.add_safe_globals([pathlib.PosixPath])
with torch.serialization.safe_globals([pathlib.PosixPath]):
    # Mock sys.argv to pass arguments to the main function of the export module
    sys.argv = ['{export_mod}', '/content/project/model.ckpt', '{output_arg}']
    module = importlib.import_module('{export_mod}')
    module.main()
"""
    with open("piper_wrapper.py", "w") as f:
        f.write(wrapper_script)

    !export TORCH_FORCE_WEIGHTS_ONLY_LOAD=0 && python3 piper_wrapper.py

    print("\033[93mCompressing...")
    !tar -czvf "{packages_path}/{export_voice_name}.tar.gz" -C "{export_voice_path}" .
    output.eval_js(f'new Audio("{guideurl}/success.wav?raw=true").play()')
    print("\033[93mDone!")

export_voice_name = f"{language}-{voice_name}{'+RT' if streaming else ''}-{quality}"
export_voice_path = f"/content/project/voice-{export_voice_name}"
packages_path = "/content/project/packages"
os.makedirs(export_voice_path, exist_ok=True)
os.makedirs(packages_path, exist_ok=True)

print("\033[93mDownloading files...")
m_id = get_drive_id(model_id)
c_id = get_drive_id(config_id)

!gdown "{m_id}" -O /content/project/model.ckpt
!gdown "{c_id}" -O "{export_voice_path}/{export_voice_name}.onnx.json"

if os.path.exists(f"{export_voice_path}/{export_voice_name}.onnx.json") and streaming:
    with open(f"{export_voice_path}/{export_voice_name}.onnx.json", "r+") as f:
        data = json.load(f)
        data.update({"streaming": True, "key": export_voice_name})
        f.seek(0)
        json.dump(data, f, indent=4)
        f.truncate()

if not write_model_card:
    start_process(streaming)
else:
    print("Model card writing enabled...")

Streaming output truncated to the last 5000 lines.
    
    # File: /usr/local/lib/python3.12/dist-packages/torch/nn/modules/conv.py:375 in forward, code: return self._conv_forward(input, self.weight, self.bias)
    conv1d_36: "f32[s31, 384, s12]" = torch.ops.aten.conv1d.default(mul_76, arg109_1, arg110_1);  arg109_1 = arg110_1 = None
    
    # File: /content/piper/src/python/piper_train/vits/models.py:206 in forward, code: stats = self.proj(x) * x_mask
    mul_77: "f32[s31, 384, s12]" = torch.ops.aten.mul.Tensor(conv1d_36, type_as);  conv1d_36 = None
    
    # File: /content/piper/src/python/piper_train/vits/models.py:208 in forward, code: m, logs = torch.split(stats, self.out_channels, dim=1)
    split = torch.ops.aten.split.Tensor(mul_77, 192, 1);  mul_77 = None
    getitem: "f32[s31, 192, s12]" = split[0];  getitem = None
    getitem_1: "f32[s31, 192, s12]" = split[1];  split = getitem_1 = None
    
    # File: /content/piper/src/python/piper_train/vits/models.py:64 in forward, c

In [ ]:
#@markdown # <font color="ffc800"> **Download/export your generated voice package.** 📥
#@markdown ---

#@markdown #### *How do you want to export your model?*
export_mode = "Download the voice package on my device (may take some time)" #@param ["Download the voice package on my device (may take some time)", "upload it to my Google Drive"]
print("\033[93mExporting package...")
if export_mode == "Download the voice package on my device (may take some time)":
    from google.colab import files
    files.download(f"{packages_path}/{export_voice_name}.tar.gz")
    msg = "Please wait a moment while the package is being downloaded."
else:
    voicepacks_folder = "/content/drive/MyDrive/piper voice packages"
    from google.colab import drive
    drive.mount('/content/drive')
    if not os.path.exists(voicepacks_folder):
        os.makedirs(voicepacks_folder)
    !cp "{packages_path}/{export_voice_name}.tar.gz" "{voicepacks_folder}"
    msg = f"You can find the generated voice package at: {voicepacks_folder}."
print(f"\033[93mDone! {msg}")

# "*I want to test this model! I don't need anything else anymore?*"

No, this is almost the end! Now you can share your generated package to your friends, upload to a cloud storage and/or test it on:
* [The inference notebook](https://colab.research.google.com/github/rmcpantoja/piper/blob/master/notebooks/piper_inference_(ONNX).ipynb)
  * run the cells in order for it to work correctly, as well as all the notebooks. Also, the inference notebook will guide you through the process using the enhanced accessibility feature if you wish. It's easy to use. Test it!
* Or through the NVDA screen reader!
  * Download and install the latest version of the [add-on](https://github.com/mush42/piper-nvda/releases).
  * Once the add-on is installed, go to NVDA menu/piper voice manager...
  * In the installed voices page, tab until you find the `Install from local file` button, press enter and select the generated package in your downloads.
  * Once the package is selected and installed, apply the changes and restart NVDA to update the voice list.
* Enjoy your creation!